In [11]:
import torch
import torch.nn as nn
class PatchEmbedding(nn.Module):
    def __init__(self,img_size=224,in_channel=3,patch_size=16,embed_size=768):
        super().__init__()
        self.proj=nn.Conv2d(in_channels=in_channel,out_channels=embed_size,kernel_size=patch_size,stride=patch_size)
    def forward(self,x):
        B,C,H,W=x.shape
        x=self.proj(x).flatten(2).transpose(1,2) # Shape = Bx768x14x14
        return x
class PositionalEmbbeding(nn.Module):
    def __init__(self,embed_size,num_token):
        super().__init__()
        self.pos_vector=nn.Parameter(torch.randn(1,num_token+1,embed_size))
    def forward(self,x):
        return x+self.pos_vector
class MultiheadAttention(nn.Module):
    def __init__(self,embed_dim,num_head):
        super().__init__()
        self.attention=nn.MultiheadAttention(embed_dim=embed_dim,num_heads=num_head)
    def forward(self,x):
        return self.attention(x,x,x)[0]
class EncoderBlock(nn.Module):
    def __init__(self,embed_dim=768,mlp_dim=1024,num_head=8):
        super().__init__()
        self.attn =  MultiheadAttention(embed_dim, num_head)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.ReLU(),
            nn.Linear(mlp_dim, embed_dim)
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x
class VisionTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=16, num_classes=10, embed_dim=768, num_heads=8, depth=6, mlp_dim=1024):
        super().__init__()
        self.patch_embedding = PatchEmbedding(img_size, 3,patch_size, embed_dim)
        self.pos_encoding = PositionalEmbbeding(embed_dim, (img_size // patch_size) ** 2)
        self.transformer_blocks = nn.ModuleList([
            EncoderBlock(embed_dim, mlp_dim, num_head=num_heads) for _ in range(depth)
        ])
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.mlp_head = nn.Linear(embed_dim, num_classes)
    def forward(self, x):
        B = x.size(0)
        x = self.patch_embedding(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = self.pos_encoding(x)
        for block in self.transformer_blocks:
            x = block(x)
        return self.mlp_head(x[:, 0])

In [19]:
model = VisionTransformer()
x=torch.randn(1,3,224,224)
y=model(x)
p=nn.Softmax(y)
Prediction=torch.argmax(y,1)
print(f"Output shape: {y.shape}")
print(f"Output value: {y}")
print(f"Output prediction: {Prediction}")

Output shape: torch.Size([1, 10])
Output value: tensor([[-0.4866,  0.8640, -0.3910,  1.1712,  0.8512, -0.5371,  1.5343,  0.4058,
          1.6960, -0.4095]], grad_fn=<AddmmBackward0>)
Output prediction: tensor([8])
